# <font color="Green">**Notebook Purpose**</font>

This notebook builds **eFigure 16 — Distribution of Higher-Level Cluster Categories by Race/Ethnicity and by HF/CKD Status**, added in response to Reviewer 5.

**Panel A:** 100% stacked bar chart showing how each race/ethnicity group distributes across the 5 higher-level prescribing trajectory categories.

**Panel B:** 100% stacked bar chart showing the same distribution stratified by composite HF/CKD status (Any HF or CKD vs. Neither) before June 1, 2019.

The same color scheme is used in both panels so they can be read together.

**Inputs (same `/content/` convention as other notebooks):**
- `monotherapy_patients.pkl`, `dual_therapy_patients.pkl`, `complex_therapy_patients.pkl`, `variant_therapy_patients.pkl`, `GLP_1_therapy_patients.pkl` — each maps cluster_id → patient_id list; together these define the 30-cluster primary analytic cohort (early-disengagement clusters are NOT loaded, so they are correctly excluded by construction).
- `patient_demographics.csv` — for race/ethnicity.
- `patient_comorbidities.csv` — for HF/CKD composite status.

**Outputs:** `eFigure16_distribution_by_race_and_hfckd.png` (300 dpi) and `eFigure16_distribution_by_race_and_hfckd.pdf` for publication.


## Section 1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import pickle as pkl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## Section 2 — Load Data

Load the 5 therapy-group pickle files. Early disengagement patients are **not** loaded, so the primary analytic cohort (n ≈ 9,327) is established by construction without needing a separate exclusion step.

In [ ]:
# 5 higher-level categories, in canonical manuscript order (matches eTable 9, eTable 10)
CATEGORIES = ['Monotherapy', 'Dual Therapy', 'Complex Therapy', 'Variant Therapy', 'GLP-1 Therapy']

PICKLE_PATHS = {
    'Monotherapy':     '/content/monotherapy_patients.pkl',
    'Dual Therapy':    '/content/dual_therapy_patients.pkl',
    'Complex Therapy': '/content/complex_therapy_patients.pkl',
    'Variant Therapy': '/content/variant_therapy_patients.pkl',
    'GLP-1 Therapy':   '/content/GLP_1_therapy_patients.pkl',
}

group_dicts = {}
for cat, path in PICKLE_PATHS.items():
    with open(path, 'rb') as f:
        group_dicts[cat] = pkl.load(f)
    n_clusters = len(group_dicts[cat])
    n_patients = sum(len(v) for v in group_dicts[cat].values())
    print(f'{cat}: {n_clusters} clusters, {n_patients:,} patients')

# Demographics & comorbidities
patient_demographics = pd.read_csv('/content/patient_demographics.csv')
patient_comorbidities = pd.read_csv('/content/patient_comorbidities.csv')
patient_comorbidities['date'] = pd.to_datetime(patient_comorbidities['date'], errors='coerce')

print()
print(f'Demographics rows: {len(patient_demographics):,}')
print(f'Comorbidities rows: {len(patient_comorbidities):,}')

## Section 3 — Build `patient_id → category` Mapping

Flatten the 5 group dictionaries into a single lookup. The set of keys defines the primary analytic cohort used for this figure.

In [ ]:
pid_to_category = {}
for cat, cluster_dict in group_dicts.items():
    for cluster_id, pid_list in cluster_dict.items():
        for pid in pid_list:
            pid_to_category[str(pid)] = cat

# Sanity check
print(f'Total primary-cohort patients: {len(pid_to_category):,}')
print()
print('Patients per category:')
for cat in CATEGORIES:
    n = sum(1 for v in pid_to_category.values() if v == cat)
    print(f'  {cat}: {n:,} ({100*n/len(pid_to_category):.1f}%)')

## Section 4 — Assemble Per-Patient Panel Attributes

For every patient in the primary cohort, attach:
- their higher-level category (Section 3)
- their race/ethnicity (already harmonized in `patient_demographics['race/ethnicity']`)
- their composite HF/CKD status (HF OR CKD documented between Jan 1 and June 1, 2019)

This produces one row per patient with everything the figure needs.

In [ ]:
# Race/ethnicity (display order matches Table 1 and eTable 11)
RACE_INPUT_TO_DISPLAY = {
    'White':    'White, NH',
    'Black':    'African American/Black, NH',
    'Hispanic': 'Hispanic/Latinx',
    'Asian':    'Asian, NH',
    'Other':    'Other, NH',
}
RACE_ORDER = list(RACE_INPUT_TO_DISPLAY.values())

dem = patient_demographics.copy()
dem['patient_id'] = dem['patient_id'].astype(str)
dem = dem[dem['patient_id'].isin(pid_to_category)].copy()
dem['category'] = dem['patient_id'].map(pid_to_category)
dem['race_display'] = dem['race/ethnicity'].astype(str).str.strip().map(RACE_INPUT_TO_DISPLAY)

# Composite HF/CKD before June 1, 2019 (same definition as eTable 11)
WINDOW_END = pd.Timestamp('2019-06-01')
com = patient_comorbidities.copy()
com['patient_id'] = com['patient_id'].astype(str)
com_in_window = com[com['date'] <= WINDOW_END].copy()

for col in ['HF', 'CKD']:
    com_in_window[col] = com_in_window[col].astype(str).str.strip().str.upper().isin(
        ['TRUE', '1', 'T', 'Y', 'YES']
    )

hfckd_any_ids = set(
    com_in_window.loc[com_in_window['HF'] | com_in_window['CKD'], 'patient_id'].unique()
)
dem['hfckd_status'] = np.where(
    dem['patient_id'].isin(hfckd_any_ids),
    'Any HF or CKD',
    'Neither HF nor CKD'
)

# Quick sanity check
print('Patients per race/ethnicity:')
print(dem['race_display'].value_counts().reindex(RACE_ORDER))
print()
print('Patients per HF/CKD status:')
print(dem['hfckd_status'].value_counts())
print()
print(f'Final patient rows for the figure: {len(dem):,}')

## Section 5 — Compute Distributions

For each panel, compute a (stratum × category) matrix of **column-wise percentages** (each stratum sums to 100% across categories).

In [ ]:
def dist_matrix(df, stratum_col, stratum_order, category_col='category', category_order=CATEGORIES):
    """Return DataFrame of percentages: index=stratum, columns=category, rows sum to 100."""
    counts = (df.groupby([stratum_col, category_col]).size()
                .unstack(category_col, fill_value=0)
                .reindex(index=stratum_order, columns=category_order, fill_value=0))
    pct = counts.div(counts.sum(axis=1), axis=0) * 100
    return counts, pct

# Panel A: by race/ethnicity
race_counts, race_pct = dist_matrix(dem, 'race_display', RACE_ORDER)
print('Panel A (race/ethnicity) -- patient counts:')
print(race_counts)
print()
print('Panel A -- percentages:')
print(race_pct.round(1))
print()

# Panel B: by HF/CKD status
HFCKD_ORDER = ['Neither HF nor CKD', 'Any HF or CKD']
hfckd_counts, hfckd_pct = dist_matrix(dem, 'hfckd_status', HFCKD_ORDER)
print('Panel B (HF/CKD) -- patient counts:')
print(hfckd_counts)
print()
print('Panel B -- percentages:')
print(hfckd_pct.round(1))

## Section 6 — Build the Figure

Two-panel layout with a shared legend at the top. Category color order matches the manuscript narrative (Monotherapy → Dual → Complex → Variant → GLP-1). Each bar is annotated with its denominator under the x-axis tick label, and each stack segment is labeled in-place with its percentage.

In [ ]:
# Colorblind-aware categorical palette (5 distinct hues; same order as CATEGORIES)
CATEGORY_COLORS = {
    'Monotherapy':     '#4477AA',  # blue
    'Dual Therapy':    '#66CCEE',  # cyan
    'Complex Therapy': '#EE6677',  # rose
    'Variant Therapy': '#CCBB44',  # gold
    'GLP-1 Therapy':   '#228833',  # green
}

PCT_LABEL_THRESHOLD = 4.0  # only annotate segments >= this %, to avoid crowding

def stacked_panel(ax, pct_df, count_df, title, xlabel_extra=None):
    strata = list(pct_df.index)
    x = np.arange(len(strata))
    bottoms = np.zeros(len(strata))

    for cat in CATEGORIES:
        heights = pct_df[cat].values
        ax.bar(x, heights, bottom=bottoms, color=CATEGORY_COLORS[cat],
               edgecolor='white', linewidth=0.5, label=cat)
        # In-bar percentage annotations
        for xi, (h, b) in enumerate(zip(heights, bottoms)):
            if h >= PCT_LABEL_THRESHOLD:
                ax.text(xi, b + h / 2, f'{h:.1f}%', ha='center', va='center',
                        fontsize=8, color='white', fontweight='bold')
        bottoms += heights

    # X-axis: stratum label + denominator
    tick_labels = []
    for s in strata:
        n = int(count_df.loc[s].sum())
        tick_labels.append(f'{s}\n(N={n:,})')
    ax.set_xticks(x)
    ax.set_xticklabels(tick_labels, fontsize=10)

    ax.set_ylim(0, 100)
    ax.set_yticks(np.arange(0, 101, 20))
    ax.set_yticklabels([f'{v}%' for v in np.arange(0, 101, 20)], fontsize=9)
    ax.set_ylabel('Percentage of patients within stratum', fontsize=10)

    ax.set_title(title, fontsize=12, fontweight='bold', pad=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Build figure: 2 panels side by side
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 6),
                                gridspec_kw={'width_ratios': [5, 2]})

stacked_panel(axA, race_pct, race_counts,
              title='A. Distribution by Race/Ethnicity')
stacked_panel(axB, hfckd_pct, hfckd_counts,
              title='B. Distribution by HF/CKD Status')

# Shared legend at the top
handles = [mpatches.Patch(facecolor=CATEGORY_COLORS[c], edgecolor='white', label=c)
           for c in CATEGORIES]
fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 1.02),
           ncol=5, frameon=False, fontsize=10, columnspacing=2.0)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## Section 7 — Save Outputs

Save both PNG (300 dpi raster) and PDF (vector) versions of the figure for the supplement.

In [ ]:
OUT_PNG = '/content/eFigure16_distribution_by_race_and_hfckd.png'
OUT_PDF = '/content/eFigure16_distribution_by_race_and_hfckd.pdf'

fig.savefig(OUT_PNG, dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(OUT_PDF,            bbox_inches='tight', facecolor='white')

print(f'Wrote {OUT_PNG}')
print(f'Wrote {OUT_PDF}')

## Section 8 — Suggested eFigure 16 Caption

> **eFigure 16. Distribution of Higher-Level Prescribing Trajectory Categories by Race/Ethnicity and by HF/CKD Status.**
> Each bar shows the percentage of patients within a given stratum (race/ethnicity in Panel A; composite HF/CKD status in Panel B) assigned to each of the five higher-level prescribing trajectory categories defined in eTable 9 (Monotherapy, Dual Therapy, Complex Therapy, Variant Therapy, GLP-1 Therapy). Bars sum to 100% within each stratum. Stratum-level denominators are shown beneath each x-axis label. The composite HF/CKD indicator follows the same definition used elsewhere in the analysis (documented heart failure or chronic kidney disease between January 1 and June 1, 2019). Early-disengagement clusters are excluded by construction. **Abbreviations:** CKD, chronic kidney disease; HF, heart failure; NH, Non-Hispanic.
